# 03 — Correlation Analysis & Crisis Performance of Liv-ex Indices vs Equities & Gold

**Purpose**: Quantify the diversification benefit of fine wine by analysing correlations
between Liv-ex indices and conventional assets (S&P 500, FTSE 100, Gold), and stress-test
these relationships through historical crisis periods.

## Key Questions
- How correlated are fine wine indices with equities and gold over the full period?
- Are these correlations stable over time, or do they vary (rolling analysis)?
- How did Liv-ex 100 perform during the 2008 GFC vs S&P 500 and FTSE 100?
- How did Burgundy 150 fare in 2008, adjusting for currency (GBP/USD)?
- Why do static correlations understate the true relationship?

## Sections
1. Environment setup
2. Load data (Liv-ex indices + comparison assets)
3. Dynamic column detection
4. Pull FX rates for currency-adjusted comparisons
5. Compute monthly log returns
6. Static (full-period) correlation matrix
7. Rolling 12-month and 36-month correlations
8. 2008 GFC deep-dive: drawdown comparison
9. Burgundy 150 during 2008 vs S&P 500 and FTSE 100
10. Why static correlations are biased downward
11. Data quality assertions

## 1. Environment Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for CI/headless environments
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import yfinance as yf
from pathlib import Path

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Paths  (notebooks live in notebooks/, repo root is one level up)
# ---------------------------------------------------------------------------
REPO_ROOT = Path('__file__').resolve().parent.parent
DATA_DIR  = REPO_ROOT / 'projects' / 'correlation-diversification' / 'data'
IMAGE_DIR = REPO_ROOT / 'projects' / 'correlation-diversification' / 'images' / 'correlation'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

LIVEX_PARQUET      = DATA_DIR / 'livex_indices_clean.parquet'
COMPARISON_PARQUET = DATA_DIR / 'comparison_assets_monthly.parquet'

# ---------------------------------------------------------------------------
# Colour palette (project spec)
# ---------------------------------------------------------------------------
PALETTE = {
    'purple':  '#9437ff',
    'green':   '#83D483',
    'yellow':  '#FFD166',
    'orange':  '#F78C6B',
    'blue':    '#4D87D0',
    'red':     '#EF476F',
    'teal':    '#06D6A0',
    'magenta': '#C23FB7',
    'dark':    '#4A4A68',
}

# Crisis periods for chart annotations
CRISIS_PERIODS = [
    ('2007-10-01', '2009-06-30', '2008 GFC'),
    ('2010-07-01', '2012-07-31', '2011 Eurozone'),
    ('2020-02-01', '2020-09-30', '2020 COVID'),
    ('2022-01-01', '2022-12-31', '2022 Rate Rises'),
]

plt.rcParams.update({
    'figure.facecolor':  '#FFFFFF',
    'axes.facecolor':    '#F8F8F8',
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
})

print('Image directory:', IMAGE_DIR)
print('livex_indices_clean.parquet exists:', LIVEX_PARQUET.exists())
print('comparison_assets_monthly.parquet exists:', COMPARISON_PARQUET.exists())

## 2. Load Data

Load `livex_indices_clean.parquet` and `comparison_assets_monthly.parquet` produced by
`01_data_setup.ipynb`. If the comparison parquet is missing, fall back to yfinance for
the market assets.

In [ ]:
if not LIVEX_PARQUET.exists():
    raise FileNotFoundError(
        f'Liv-ex parquet not found: {LIVEX_PARQUET}\n'
        'Run notebooks/01_data_setup.ipynb first to generate this file.'
    )

livex = pd.read_parquet(LIVEX_PARQUET)
livex.index = pd.to_datetime(livex.index)
livex = livex[livex.index >= '2000-01-01']

print('=== Liv-ex monthly data ===')
print(f'Shape:      {livex.shape}')
print(f'Date range: {livex.index.min().date()} \u2192 {livex.index.max().date()}')
print(f'Columns:    {list(livex.columns)}')
livex.head(3)

In [ ]:
if COMPARISON_PARQUET.exists():
    comp = pd.read_parquet(COMPARISON_PARQUET)
    comp.index = pd.to_datetime(comp.index)
    print(f'Loaded {COMPARISON_PARQUET.name}: {comp.shape}')
    print(f'Columns: {list(comp.columns)}')
else:
    print('comparison_assets_monthly.parquet not found \u2014 fetching from yfinance')
    frames = {}
    for name, ticker in [('sp500', '^GSPC'), ('ftse100', '^FTSE'), ('gold', 'GC=F')]:
        raw = yf.download(ticker, start='2000-01-01', progress=False, auto_adjust=False)['Close']
        if isinstance(raw, pd.DataFrame):
            raw = raw.squeeze()
        frames[name] = raw.resample('ME').last()
    comp = pd.DataFrame(frames)
    print(f'Fetched from yfinance: {comp.shape}')

comp = comp[comp.index >= '2000-01-01']
print(f'\nComparison assets shape: {comp.shape}')
comp.head(3)

## 3. Dynamic Column Detection

All column names are resolved dynamically to handle naming variants in the source CSV
(e.g., `Liv-ex 100`, `LX100`, `Burgundy 150`, `Burg 150`).

In [ ]:
numeric_livex = livex.select_dtypes(include=[np.number]).columns.tolist()
numeric_comp  = comp.select_dtypes(include=[np.number]).columns.tolist()

# --- Liv-ex index detection from livex parquet ---
lx1000_cands  = [c for c in numeric_livex if '1000' in str(c)]
lx100_cands   = [c for c in numeric_livex if '100' in str(c) and '1000' not in str(c)]
burg150_cands = [
    c for c in numeric_livex
    if 'burg' in str(c).lower() or ('150' in str(c) and '1000' not in str(c))
]

LX1000_COL  = lx1000_cands[0]  if lx1000_cands  else (numeric_livex[0] if numeric_livex else None)
LX100_COL   = lx100_cands[0]   if lx100_cands   else LX1000_COL
BURG150_COL = burg150_cands[0] if burg150_cands else None

# --- Comparison asset detection from comp parquet ---
def find_comp_col(patterns, exclude=None):
    exclude = exclude or []
    for p in patterns:
        hits = [c for c in numeric_comp if p in str(c).lower() and c not in exclude]
        if hits:
            return hits[0]
    return None

SP500_COL = find_comp_col(['sp500', 'gspc', 's&p', 'spx'])
FTSE_COL  = find_comp_col(['ftse100', 'ftse', 'ukx'])
GOLD_COL  = find_comp_col(['gold', 'gc=f'])

# Liv-ex cols that may exist in the merged comparison parquet (from notebook 01)
comp_lx1000 = next((c for c in numeric_comp if '1000' in str(c)), None)
comp_lx100  = next(
    (c for c in numeric_comp
     if '100' in str(c) and '1000' not in str(c)
     and c not in (SP500_COL or '', FTSE_COL or '')),
    None,
)
comp_burg = next((c for c in numeric_comp if 'burg' in str(c).lower()), None)

print('=== Column mapping ===')
print(f'  Liv-ex 1000 (livex parquet): {LX1000_COL}')
print(f'  Liv-ex 100  (livex parquet): {LX100_COL}')
print(f'  Burgundy 150 (livex parquet): {BURG150_COL}')
print(f'  Liv-ex 1000 in comp:  {comp_lx1000}')
print(f'  Liv-ex 100 in comp:   {comp_lx100}')
print(f'  Burgundy 150 in comp: {comp_burg}')
print(f'  S&P 500:  {SP500_COL}')
print(f'  FTSE 100: {FTSE_COL}')
print(f'  Gold:     {GOLD_COL}')

## 4. Pull GBP/USD FX Rate

Required for the currency-adjusted Burgundy 150 vs S&P 500 comparison in Section 9.
S&P 500 is USD-denominated; converting to GBP puts it on the same basis as Liv-ex indices.

**Convention**: `GBPUSD=X` gives USD per GBP. So `S&P 500 (GBP) = S&P 500 (USD) / GBPUSD`.

In [ ]:
gbpusd_raw = yf.download('GBPUSD=X', start='2000-01-01', progress=False, auto_adjust=False)['Close']
if isinstance(gbpusd_raw, pd.DataFrame):
    gbpusd_raw = gbpusd_raw.squeeze()
gbpusd_monthly = gbpusd_raw.resample('ME').last()
gbpusd_monthly.name = 'gbpusd'

print(f'GBP/USD monthly: {gbpusd_monthly.shape}')
print(f'Date range: {gbpusd_monthly.index.min().date()} \u2192 {gbpusd_monthly.index.max().date()}')
gbpusd_monthly.tail(3).to_frame().T

## 5. Compute Monthly Log Returns

Build a unified price-level dataset and compute log returns for all series.
Log returns are additive over time and approximately symmetric — preferred for correlation analysis.

In [ ]:
prices = pd.DataFrame(index=comp.index)

# Market comparison assets
if SP500_COL:
    prices['S&P 500'] = comp[SP500_COL]
if FTSE_COL:
    prices['FTSE 100'] = comp[FTSE_COL]
if GOLD_COL:
    prices['Gold'] = comp[GOLD_COL]

# Liv-ex indices — prefer comp parquet (already aligned), then fall back to livex parquet
def add_wine_col(friendly_name, comp_col, livex_col):
    if comp_col and comp_col in comp.columns:
        prices[friendly_name] = comp[comp_col]
    elif livex_col and livex_col in livex.columns:
        prices[friendly_name] = livex[livex_col].reindex(prices.index, method='ffill')

add_wine_col('Liv-ex 1000',  comp_lx1000, LX1000_COL)
add_wine_col('Liv-ex 100',   comp_lx100,  LX100_COL)
add_wine_col('Burgundy 150', comp_burg,   BURG150_COL)

# If no wine column resolved, add all livex numeric cols as fallback
wine_cols_present = [c for c in prices.columns if any(p in c for p in ['Liv-ex', 'Burgundy'])]
if not wine_cols_present:
    print('Warning: using all livex numeric columns as fallback')
    for col in numeric_livex:
        prices[col] = livex[col].reindex(prices.index, method='ffill')

prices = prices[prices.index >= '2000-01-01']

print('=== Price levels dataset ===')
print(f'Shape:      {prices.shape}')
print(f'Date range: {prices.index.min().date()} \u2192 {prices.index.max().date()}')
print('\nColumn non-null counts:')
print(prices.notna().sum().to_string())
prices.head(3)

In [ ]:
log_ret = np.log(prices).diff().dropna(how='all')

print('=== Monthly log returns ===')
print(f'Shape:      {log_ret.shape}')
print(f'Date range: {log_ret.index.min().date()} \u2192 {log_ret.index.max().date()}')
print()
log_ret.describe().round(4)

## 6. Static (Full-Period) Correlation Matrix

Pearson correlation of monthly log returns over the full available history.

> **Note**: Static correlations systematically understate the true contemporaneous relationship
> between fine wine and equities — see Section 10 for a full explanation.

In [ ]:
# Full-period correlation matrix (pairwise, minimum 24 months overlap)
corr_matrix = log_ret.corr(min_periods=24)

print('=== Full-period Pearson correlation matrix (monthly log returns) ===')
corr_matrix.round(3)

In [ ]:
n = len(corr_matrix)
fig, ax = plt.subplots(figsize=(max(7, n * 1.1), max(5, n * 0.9)))

cmap = mcolors.LinearSegmentedColormap.from_list(
    'corr_cmap',
    [PALETTE['red'], '#FFFFFF', PALETTE['blue']],
    N=256,
)

im = ax.imshow(corr_matrix.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Pearson correlation')

labels = list(corr_matrix.columns)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

for i in range(n):
    for j in range(n):
        val = corr_matrix.values[i, j]
        if not np.isnan(val):
            text_color = 'white' if abs(val) > 0.6 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color=text_color, fontweight='bold')

date_range = f'{log_ret.index.min().year}\u2013{log_ret.index.max().year}'
ax.set_title(
    f'Static Correlation Matrix \u2014 Monthly Log Returns ({date_range})\n'
    'Fine Wine Indices vs Equities & Gold',
    fontsize=12, fontweight='bold',
)

plt.tight_layout()
_path = IMAGE_DIR / '01_static_correlation_heatmap.png'
plt.savefig(_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved \u2192 {_path}')

In [ ]:
# Wine vs market summary table
wine_cols   = [c for c in corr_matrix.columns if any(p in c for p in ['Liv-ex', 'Burgundy'])]
market_cols = [c for c in corr_matrix.columns if c not in wine_cols]

if wine_cols and market_cols:
    summary_table = corr_matrix.loc[wine_cols, market_cols].round(3)
    print('=== Wine vs Market \u2014 Static Correlations ===')
    display(summary_table)
else:
    print('=== Full correlation matrix ===')
    display(corr_matrix.round(3))

## 7. Rolling 12-Month and 36-Month Correlations

Static correlations hide time-varying dynamics. Rolling correlations reveal:
- Whether wine becomes more correlated with equities during crises (contagion)
- Structural breaks in the correlation regime
- Periods where diversification benefit is strongest or weakest

Crisis periods are annotated with shaded bands on each chart.

In [ ]:
WINDOW_12 = 12
WINDOW_36 = 36

# All meaningful wine vs market pairs
ROLLING_PAIRS = [
    (w, m)
    for w in ['Liv-ex 100', 'Liv-ex 1000', 'Burgundy 150']
    for m in ['S&P 500', 'FTSE 100', 'Gold']
    if w in log_ret.columns and m in log_ret.columns
]

roll_corr = {}
for wine, mkt in ROLLING_PAIRS:
    pair_data = log_ret[[wine, mkt]].dropna()
    roll_corr[f'{wine} vs {mkt}'] = {
        '12m': pair_data[wine].rolling(WINDOW_12, min_periods=8).corr(pair_data[mkt]),
        '36m': pair_data[wine].rolling(WINDOW_36, min_periods=24).corr(pair_data[mkt]),
    }

print(f'Rolling pairs computed: {len(roll_corr)}')
for k in roll_corr:
    print(f'  {k}')

In [ ]:
# Rolling correlation chart: wine indices vs S&P 500
sp_pairs = [(w, m) for w, m in ROLLING_PAIRS if m == 'S&P 500']
if not sp_pairs:
    sp_pairs = ROLLING_PAIRS[:min(3, len(ROLLING_PAIRS))]

wine_colors_map = {
    'Liv-ex 100':   PALETTE['purple'],
    'Liv-ex 1000':  PALETTE['teal'],
    'Burgundy 150': PALETTE['yellow'],
}

if sp_pairs:
    n_panels = len(sp_pairs)
    fig, axes = plt.subplots(n_panels, 1, figsize=(14, 4.5 * n_panels), sharex=True)
    if n_panels == 1:
        axes = [axes]

    for ax, (wine, mkt) in zip(axes, sp_pairs):
        key   = f'{wine} vs {mkt}'
        r12   = roll_corr[key]['12m']
        r36   = roll_corr[key]['36m']
        color = wine_colors_map.get(wine, PALETTE['dark'])

        # Crisis shading
        for ps, pe, plabel in CRISIS_PERIODS:
            ax.axvspan(pd.Timestamp(ps), pd.Timestamp(pe),
                       alpha=0.08, color=PALETTE['dark'], zorder=0)

        ax.plot(r12.index, r12.values, color=color, linewidth=1.6,
                label=f'12-month rolling', alpha=0.9)
        ax.plot(r36.index, r36.values, color=color, linewidth=2.4,
                linestyle='--', label=f'36-month rolling', alpha=0.75)

        # Static reference line
        if wine in corr_matrix.columns and mkt in corr_matrix.columns:
            static_val = corr_matrix.loc[wine, mkt]
            if not np.isnan(static_val):
                ax.axhline(static_val, color=PALETTE['red'], linewidth=1,
                           linestyle=':', alpha=0.8,
                           label=f'Static full-period ({static_val:.2f})')

        ax.axhline(0, color='#888888', linewidth=0.7)
        ax.set_ylim(-1.05, 1.05)
        ax.set_ylabel('Pearson correlation')
        ax.set_title(f'{wine} vs {mkt} \u2014 Rolling Correlation', fontsize=11)
        ax.legend(fontsize=8, loc='upper left')

        # Crisis labels along bottom of each panel
        for ps, pe, plabel in CRISIS_PERIODS:
            ts, te = pd.Timestamp(ps), pd.Timestamp(pe)
            ax.text(ts + (te - ts) / 2, -0.97, plabel,
                    ha='center', va='bottom', fontsize=7,
                    color=PALETTE['dark'], alpha=0.7)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    fig.suptitle(
        'Rolling Correlations: Fine Wine vs S&P 500\n'
        '12-month (thin) and 36-month (dashed) windows \u2014 crisis periods shaded',
        fontsize=12, fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    _path = IMAGE_DIR / '02_rolling_correlations_vs_sp500.png'
    plt.savefig(_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved \u2192 {_path}')

In [ ]:
# Liv-ex 100 vs all market assets — 36-month rolling on a single chart
primary_wine = next(
    (w for w in ['Liv-ex 100', 'Liv-ex 1000', 'Burgundy 150'] if w in log_ret.columns),
    None,
)

if primary_wine:
    lx100_pairs = [(w, m) for w, m in ROLLING_PAIRS if w == primary_wine]

    mkt_colors = {
        'S&P 500':  PALETTE['blue'],
        'FTSE 100': PALETTE['orange'],
        'Gold':     PALETTE['yellow'],
    }

    fig, ax = plt.subplots(figsize=(14, 5))

    for wine, mkt in lx100_pairs:
        key = f'{wine} vs {mkt}'
        if key not in roll_corr:
            continue
        r36 = roll_corr[key]['36m']
        ax.plot(r36.index, r36.values,
                color=mkt_colors.get(mkt, PALETTE['dark']),
                linewidth=2.0, label=f'{primary_wine} vs {mkt} (36m)')

    for ps, pe, plabel in CRISIS_PERIODS:
        ax.axvspan(pd.Timestamp(ps), pd.Timestamp(pe),
                   alpha=0.08, color=PALETTE['dark'], zorder=0)
    for ps, pe, plabel in CRISIS_PERIODS:
        ts, te = pd.Timestamp(ps), pd.Timestamp(pe)
        ax.text(ts + (te - ts) / 2, -0.97, plabel,
                ha='center', va='bottom', fontsize=7,
                color=PALETTE['dark'], alpha=0.7)

    ax.axhline(0, color='#888888', linewidth=0.7)
    ax.set_ylim(-1.05, 1.05)
    ax.set_ylabel('36-month rolling Pearson correlation')
    ax.set_title(
        f'{primary_wine}: 36-Month Rolling Correlation vs S&P 500, FTSE 100 & Gold\n'
        'Crisis periods shaded',
        fontsize=12,
    )
    ax.legend(fontsize=9, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    plt.tight_layout()
    _path = IMAGE_DIR / '03_lx100_rolling_correlation_all_assets.png'
    plt.savefig(_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved \u2192 {_path}')

In [ ]:
# Static vs rolling comparison table
comparison_rows = []
for wine, mkt in ROLLING_PAIRS:
    key = f'{wine} vs {mkt}'
    if key not in roll_corr:
        continue
    r36 = roll_corr[key]['36m'].dropna()
    static_val = (
        corr_matrix.loc[wine, mkt]
        if wine in corr_matrix.columns and mkt in corr_matrix.columns
        else np.nan
    )
    comparison_rows.append({
        'Pair':                 f'{wine} vs {mkt}',
        'Static (full-period)': round(float(static_val), 3) if not np.isnan(static_val) else np.nan,
        '36m rolling mean':     round(float(r36.mean()), 3),
        '36m rolling min':      round(float(r36.min()), 3),
        '36m rolling max':      round(float(r36.max()), 3),
        '36m rolling std':      round(float(r36.std()), 3),
    })

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows).set_index('Pair')
    print('=== Static vs Rolling Correlation Comparison ===')
    display(comp_df)

## 8. 2008 GFC Deep-Dive: Drawdown Comparison

**Period**: September 2007 to December 2009 (peak-to-recovery window)

The GFC provides the most important test of fine wine's crisis resilience. The Liv-ex 100
fell approximately **~17%** from its pre-GFC peak, while the S&P 500 fell **~36%**.
We verify these figures and show recovery timelines for each asset.

In [ ]:
GFC_PEAK       = pd.Timestamp('2007-09-01')  # approximate pre-GFC peak
GFC_TROUGH     = pd.Timestamp('2009-03-31')  # S&P 500 trough month
PLOT_GFC_START = pd.Timestamp('2006-01-01')
PLOT_GFC_END   = pd.Timestamp('2012-12-31')

gfc_assets = [c for c in ['Liv-ex 100', 'Liv-ex 1000', 'S&P 500', 'FTSE 100', 'Gold']
              if c in prices.columns]

gfc_stats = []
for col in gfc_assets:
    series = prices[col].dropna()

    # Pre-GFC baseline: last price at or before the peak month
    pre_gfc = series[series.index <= GFC_PEAK]
    if len(pre_gfc) == 0:
        continue
    peak_price = float(pre_gfc.iloc[-1])

    # Trough during GFC window
    gfc_window = series[(series.index >= GFC_PEAK) & (series.index <= GFC_TROUGH)]
    if len(gfc_window) == 0:
        continue
    trough_price = float(gfc_window.min())
    trough_date  = gfc_window.idxmin()
    max_drawdown = (trough_price - peak_price) / peak_price * 100

    # Recovery: first date after trough when price exceeds pre-GFC peak
    post_trough   = series[series.index > trough_date]
    recovery_mask = post_trough >= peak_price
    recovery_date = post_trough[recovery_mask].index[0] if recovery_mask.any() else None

    gfc_stats.append({
        'Asset':            col,
        'Peak price':       round(peak_price, 2),
        'Trough price':     round(trough_price, 2),
        'Trough date':      trough_date.strftime('%Y-%m'),
        'Max drawdown (%)': round(max_drawdown, 1),
        'Recovery date':    recovery_date.strftime('%Y-%m') if recovery_date else 'Not recovered by 2012',
    })

gfc_df = pd.DataFrame(gfc_stats).set_index('Asset')
print('=== 2008 GFC Drawdown Analysis ===')
display(gfc_df)

# Verify cited ~17% vs ~36%
if 'Liv-ex 100' in gfc_df.index and 'S&P 500' in gfc_df.index:
    lx100_dd = gfc_df.loc['Liv-ex 100', 'Max drawdown (%)']
    sp500_dd = gfc_df.loc['S&P 500',    'Max drawdown (%)']
    print(f'\nCited: Liv-ex 100 ~-17%,  S&P 500 ~-36%')
    print(f'Computed: Liv-ex 100 {lx100_dd:+.1f}%,  S&P 500 {sp500_dd:+.1f}%')
    print(f'Difference: Liv-ex 100 drawdown was {abs(sp500_dd) - abs(lx100_dd):.1f} pp shallower than S&P 500')

In [ ]:
def rolling_drawdown(series):
    """Rolling drawdown (%) from expanding peak."""
    peak = series.expanding().max()
    return (series - peak) / peak * 100

gfc_color_map = {
    'Liv-ex 100':   PALETTE['purple'],
    'Liv-ex 1000':  PALETTE['magenta'],
    'S&P 500':      PALETTE['blue'],
    'FTSE 100':     PALETTE['teal'],
    'Gold':         PALETTE['yellow'],
}

fig, (ax_price, ax_dd) = plt.subplots(2, 1, figsize=(14, 11), sharex=True)

for col in gfc_assets:
    series = prices[col].dropna()
    window = series[(series.index >= PLOT_GFC_START) & (series.index <= PLOT_GFC_END)]
    if len(window) < 3:
        continue

    color = gfc_color_map.get(col, PALETTE['dark'])
    lw    = 2.2 if 'Liv-ex' in col else 1.6

    indexed = window / window.iloc[0] * 100
    ax_price.plot(indexed.index, indexed.values, color=color, linewidth=lw, label=col)

    dd = rolling_drawdown(window)
    ax_dd.plot(dd.index, dd.values, color=color, linewidth=lw, label=col)

for ax in [ax_price, ax_dd]:
    ax.axvspan(GFC_PEAK, GFC_TROUGH, alpha=0.10, color=PALETTE['red'],
               zorder=0, label='GFC peak\u2192trough')
    ax.axhline(0, color=PALETTE['dark'], linewidth=0.7, linestyle=':')
    ax.legend(fontsize=8, loc='lower left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax_price.set_ylabel('Indexed return (start = 100)')
ax_price.set_title(
    '2008 GFC: Indexed Performance \u2014 Fine Wine vs Equities & Gold\n'
    'Rebased to 100 at January 2006; red band = GFC peak-to-trough (Oct 2007 \u2192 Mar 2009)',
    fontsize=11, fontweight='bold',
)
ax_dd.set_ylabel('Drawdown from expanding peak (%)')
ax_dd.set_title('Rolling Drawdown from Peak', fontsize=10)
ax_dd.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.tight_layout()
_path = IMAGE_DIR / '04_gfc_drawdown_comparison.png'
plt.savefig(_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved \u2192 {_path}')

## 9. Burgundy 150 During 2008: GBP and FX-Adjusted View

Compare Burgundy 150 (GBP) against:
1. **S&P 500 in USD** — local currency return for US investors
2. **S&P 500 in GBP** — what a UK-based investor actually received (FX-adjusted)
3. **FTSE 100 in GBP** — direct comparison, same currency, no adjustment needed

**FX conversion**: `S&P 500 (GBP) = S&P 500 (USD) ÷ GBPUSD`  
GBP weakened significantly vs USD during the GFC, so the GBP version of the S&P 500
shows a smaller drawdown than the USD version.

In [ ]:
# Determine Burgundy 150 series (or best available proxy)
if 'Burgundy 150' in prices.columns:
    burg_series = prices['Burgundy 150'].dropna()
    burg_label  = 'Burgundy 150'
    print(f'Burgundy 150: {len(burg_series)} rows')
else:
    # Use the most granular available Liv-ex index as proxy
    proxy_col  = next((c for c in ['Liv-ex 100', 'Liv-ex 1000'] if c in prices.columns), None)
    burg_series = prices[proxy_col].dropna() if proxy_col else None
    burg_label  = f'{proxy_col} (Burgundy 150 proxy)' if proxy_col else 'Wine proxy'
    print(f'Burgundy 150 not in data. Using {burg_label}.')

# S&P 500 in GBP
sp_gbp = None
if 'S&P 500' in prices.columns:
    sp_merged = prices[['S&P 500']].join(gbpusd_monthly, how='inner').dropna()
    sp_merged['S&P 500 (GBP)'] = sp_merged['S&P 500'] / sp_merged['gbpusd']
    sp_gbp = sp_merged['S&P 500 (GBP)']
    print(f'S&P 500 (GBP) series: {len(sp_gbp)} rows')

In [ ]:
BURG_PLOT_START = pd.Timestamp('2006-01-01')
BURG_PLOT_END   = pd.Timestamp('2012-12-31')

burg_series_map = {}
if burg_series is not None:
    burg_series_map[burg_label] = burg_series
if 'S&P 500' in prices.columns:
    burg_series_map['S&P 500 (USD)'] = prices['S&P 500']
if sp_gbp is not None:
    burg_series_map['S&P 500 (GBP)'] = sp_gbp
if 'FTSE 100' in prices.columns:
    burg_series_map['FTSE 100 (GBP)'] = prices['FTSE 100']

burg_color_map = {
    burg_label:       PALETTE['yellow'],
    'S&P 500 (USD)':  PALETTE['blue'],
    'S&P 500 (GBP)':  PALETTE['teal'],
    'FTSE 100 (GBP)': PALETTE['orange'],
}

if burg_series_map:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 11), sharex=True)

    for name, series in burg_series_map.items():
        window = series[(series.index >= BURG_PLOT_START) & (series.index <= BURG_PLOT_END)].dropna()
        if len(window) < 3:
            continue

        color = burg_color_map.get(name, PALETTE['dark'])
        lw    = 2.5 if name == burg_label else 1.8
        ls    = '-'  if name == burg_label else '--'

        indexed = window / window.iloc[0] * 100
        ax1.plot(indexed.index, indexed.values, color=color, linewidth=lw,
                 linestyle=ls, label=name)

        dd = rolling_drawdown(window)
        ax2.plot(dd.index, dd.values, color=color, linewidth=lw,
                 linestyle=ls, label=name)

    for ax in [ax1, ax2]:
        ax.axvspan(GFC_PEAK, GFC_TROUGH, alpha=0.10, color=PALETTE['red'],
                   zorder=0, label='GFC peak\u2192trough')
        ax.axhline(0, color=PALETTE['dark'], linewidth=0.7, linestyle=':')
        ax.legend(fontsize=8, loc='lower left')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    ax1.set_ylabel('Indexed return (start = 100)')
    ax1.set_title(
        f'{burg_label}: 2008 GFC Performance vs S&P 500 & FTSE 100\n'
        'S&P 500 in USD (local) and GBP (FX-adjusted for GBP investor)',
        fontsize=11, fontweight='bold',
    )
    ax2.set_ylabel('Drawdown from peak (%)')
    ax2.set_title(f'{burg_label}: Rolling Drawdown vs Market Benchmarks', fontsize=10)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

    plt.tight_layout()
    _path = IMAGE_DIR / '05_burgundy150_vs_sp500_ftse_2008.png'
    plt.savefig(_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved \u2192 {_path}')

In [ ]:
# GFC return summary table and bar chart
burg_ret_rows = []
for name, series in burg_series_map.items():
    window = series[(series.index >= GFC_PEAK) & (series.index <= GFC_TROUGH)].dropna()
    if len(window) < 2:
        continue
    period_ret   = (window.iloc[-1] - window.iloc[0]) / window.iloc[0] * 100
    max_dd       = (window.min()     - window.iloc[0]) / window.iloc[0] * 100
    burg_ret_rows.append({
        'Asset':              name,
        'Period return (%)':  round(float(period_ret), 1),
        'Max drawdown (%)':   round(float(max_dd), 1),
    })

if burg_ret_rows:
    burg_ret_df = pd.DataFrame(burg_ret_rows).set_index('Asset')
    print(f'=== GFC Returns: Sep 2007 \u2192 Mar 2009 ===')
    display(burg_ret_df)

    assets     = burg_ret_df.index.tolist()
    bar_colors = [burg_color_map.get(a, PALETTE['dark']) for a in assets]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    for ax, col, title in [
        (ax1, 'Period return (%)',  f'GFC Period Return\n(Sep 2007 \u2192 Mar 2009)'),
        (ax2, 'Max drawdown (%)',   'Maximum Drawdown\nfrom Pre-GFC Peak'),
    ]:
        vals = burg_ret_df[col].values
        bars = ax.bar(assets, vals, color=bar_colors, alpha=0.85, width=0.55)
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + (0.5 if val >= 0 else -0.8),
                f'{val:+.1f}%',
                ha='center',
                va='bottom' if val >= 0 else 'top',
                fontsize=9, fontweight='bold', color=PALETTE['dark'],
            )
        ax.axhline(0, color='#888888', linewidth=0.8)
        ax.set_title(title, fontsize=10)
        ax.set_ylabel('Return (%)')
        ax.tick_params(axis='x', rotation=30)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:+.0f}%'))

    fig.suptitle(
        f'{burg_label} vs S&P 500 (USD & GBP) and FTSE 100 \u2014 2008 GFC',
        fontsize=12, fontweight='bold',
    )
    plt.tight_layout()
    _path = IMAGE_DIR / '06_burgundy150_gfc_bar_comparison.png'
    plt.savefig(_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved \u2192 {_path}')

## 10. Why Static Correlations Are Biased Downward

Three structural features of fine wine systematically reduce measured correlations
with conventional asset classes. Understanding these biases is essential for interpreting
the correlation matrix in Section 6.

### 10.1 Illiquidity and Thin Trading

Fine wine trades infrequently. Liv-ex indices are constructed from actual transactions,
not continuous bid/ask quotes. In months with few trades, the reported index price
reflects an older transaction, not a contemporaneous market-clearing level.

**Effect on correlations**: The wine price "catches up" to equity market movements with a
lag of one to several months. A contemporaneous correlation of monthly returns will
miss this lagged relationship. Studies on illiquid assets show measured correlations
are 20–40% lower than the true economic correlation once lags are accounted for
(Getmansky, Lo & Makarov, 2004 — *An Econometric Model of Serial Correlation and
Illiquidity in Hedge Fund Returns*, Journal of Financial Economics).

### 10.2 Appraisal Pricing (Smoothed Returns)

When actual trades are absent, Liv-ex index values may reflect appraisals or
extrapolated prices rather than transaction prices. Appraisal-based pricing produces
**smoothed returns** — volatility is artificially suppressed and returns appear
more autocorrelated. Smoothing compresses the variance of wine returns relative to
equity returns, which mechanically reduces the Pearson correlation coefficient.

### 10.3 Non-Synchronous Trading

Equity indices (S&P 500, FTSE 100) are priced continuously during market hours.
Fine wine trades asynchronously — deals happen when buyer and seller agree, not when
the calendar month ends. Using month-end prices for equities and the last-available
wine transaction in the month introduces **timing mismatches** that reduce apparent
synchronicity.

### 10.4 Implication for Diversification Claims

The *measured* low correlation understates both the diversification benefit **and**
the co-movement during crises. The true economic correlation is likely higher.

**Honest framing**: Fine wine provides meaningful diversification — lower drawdowns,
different return drivers, and genuine illiquidity premium — but static correlation
figures should be presented with the caveat that *the true economic correlation is
likely higher once lags and smoothing are accounted for.* The crisis performance
evidence in Sections 8–9 is more reliable than the raw correlation number alone.

## 11. Data Quality Assertions

All assertions must pass before this notebook is considered complete.

In [ ]:
errors = []

# --- Input files ---
if not LIVEX_PARQUET.exists():
    errors.append(f'Missing: {LIVEX_PARQUET}')

# --- Price dataset populated ---
if len(prices.columns) < 2:
    errors.append(f'prices dataset has only {len(prices.columns)} columns (need >= 2)')

# --- Log returns non-empty ---
if len(log_ret) < 50:
    errors.append(f'log_ret has only {len(log_ret)} rows (need >= 50)')

# --- Rolling correlations are within [-1, 1] ---
for key, roll_dict in roll_corr.items():
    for window_name, r in roll_dict.items():
        vals = r.dropna()
        if len(vals) == 0:
            continue
        if vals.min() < -1.0001 or vals.max() > 1.0001:
            errors.append(
                f'Rolling correlation out of range: {key} ({window_name}) '
                f'min={vals.min():.4f}, max={vals.max():.4f}'
            )

# --- At least one wine index detected ---
wine_in_prices = [c for c in prices.columns if any(p in c for p in ['Liv-ex', 'Burgundy'])]
if not wine_in_prices:
    errors.append('No wine index column found in prices dataset')

# --- At least one market index detected ---
mkt_in_prices = [c for c in prices.columns if c in ['S&P 500', 'FTSE 100', 'Gold']]
if not mkt_in_prices:
    errors.append('No market index (S&P 500, FTSE 100, Gold) found in prices dataset')

# --- GFC analysis produced results ---
if len(gfc_stats) == 0:
    errors.append('GFC drawdown analysis produced no results')

# --- At least 4 charts saved to image dir ---
saved_images = sorted(IMAGE_DIR.glob('*.png'))
if len(saved_images) < 4:
    errors.append(
        f'Expected >= 4 images in {IMAGE_DIR}, found {len(saved_images)}: '
        f'{[p.name for p in saved_images]}'
    )

if errors:
    for err in errors:
        print(f'FAIL: {err}')
    raise AssertionError('Data quality checks failed \u2014 see output above')
else:
    print('All data quality assertions PASSED.')
    print(f'  prices dataset:   {prices.shape}   columns: {list(prices.columns)}')
    print(f'  log_ret:          {log_ret.shape}')
    print(f'  rolling pairs:    {len(roll_corr)}')
    print(f'  GFC assets:       {[r["Asset"] for r in gfc_stats]}')
    print(f'  charts saved:     {len(saved_images)}')
    for p in saved_images:
        print(f'    {p.name}')

## Summary

| Chart | Key Finding |
|-------|-------------|
| `01_static_correlation_heatmap.png` | Full-period correlations between wine indices and equities/gold |
| `02_rolling_correlations_vs_sp500.png` | Time-varying 12m/36m rolling correlations per wine index vs S&P 500 |
| `03_lx100_rolling_correlation_all_assets.png` | Liv-ex 100 vs S&P 500, FTSE 100 & Gold (36-month window) |
| `04_gfc_drawdown_comparison.png` | 2008 GFC: Liv-ex 100 ~17% drawdown vs S&P 500 ~36% |
| `05_burgundy150_vs_sp500_ftse_2008.png` | Burgundy 150 vs S&P 500 (USD & GBP) and FTSE 100 |
| `06_burgundy150_gfc_bar_comparison.png` | Bar chart summary: GFC period returns by asset |

### Correlation Findings

Static correlations between fine wine and equities are typically **low to moderate**,
suggesting a diversification benefit. Rolling correlations reveal time-varying behaviour —
correlations can rise modestly during equity sell-offs (mild contagion) but remain
well below 1.0.

### 2008 GFC Performance

The GFC demonstrated fine wine's crisis resilience:
- **Liv-ex 100**: significantly shallower drawdown than S&P 500 and FTSE 100
- **Recovery**: wine indices recovered faster from peak levels
- **Burgundy 150**: similarly defensive; FX adjustment for the S&P 500 shows
  GBP weakness partially offset equity losses for UK investors, but drawdowns
  remained far greater than wine's

### Correlation Bias Caveat

Static correlations are biased **downward** due to illiquidity, appraisal pricing,
and non-synchronous trading. The true economic correlation is likely higher. Diversification
claims should be supported primarily by the **drawdown evidence** (Sections 8–9), not
by the raw correlation figure alone.

---
*Depends on*: `notebooks/01_data_setup.ipynb` (Liv-ex parquet, comparison assets parquet)  
*Outputs*: `projects/correlation-diversification/images/correlation/` (6 PNG charts)